# Modul 8: Analisis Spektral FFT & Dekomposisi Wavelet (DWT)
**Mata Kuliah:** Analisis Deret Waktu dan Peramalan (TEE-412) — S1 Teknik Elektro FT UNIB  
**Sub-CPMK 8 (Minggu 9):** Menganalisis harmonisa dan mendeteksi gangguan transien pada sinyal arus motor listrik menggunakan FFT, STFT, dan *Discrete Wavelet Transform* (DWT `db4`).

In [ ]:
# !pip install -q PyWavelets scipy matplotlib numpy
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import stft
import pywt

plt.rcParams['axes.grid'] = True

## 1. Simulasi Sinyal Arus Motor 50 Hz + Harmonisa ke-5 (250 Hz) + Transien Impuls

In [ ]:
rng = np.random.default_rng(99)
fs = 1600  # Hz (Nyquist = 800 Hz)
T = 1.0
t = np.linspace(0, T, int(fs * T), endpoint=False)

# Fundamental 50 Hz + Harmonisa ke-5 (250 Hz) + Gangguan transien pada t=0.45 s
clean_base = 10.0 * np.sin(2 * np.pi * 50 * t) + 2.5 * np.sin(2 * np.pi * 250 * t)
transient = 8.0 * np.exp(-((t - 0.45) ** 2) / (2 * 0.004 ** 2)) * np.sin(2 * np.pi * 550 * t)
noisy_signal = clean_base + transient + rng.normal(0, 0.8, len(t))

# 1. Analisis FFT
freqs = np.fft.rfftfreq(len(t), d=1 / fs)
mag = 2.0 * np.abs(np.fft.rfft(noisy_signal)) / len(t)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(t, noisy_signal, lw=0.9)
axes[0].set_title('Sinyal Arus Motor Domain Waktu (Transien di t=0.45s)')
axes[0].set_xlabel('Waktu (s)')
axes[0].set_ylabel('Arus (A)')

axes[1].plot(freqs, mag, color='crimson')
axes[1].set_xlim(0, 700)
axes[1].set_title('Spektrum FFT (Harmonisa 50 & 250 Hz Terlihat, Waktu Transien Hilang)')
axes[1].set_xlabel('Frekuensi (Hz)')
plt.tight_layout()
plt.show()

## 2. Dekomposisi DWT 3-Level (`db4`) & Soft-Thresholding Denoising

In [ ]:
# Pita DWT (fs=1600 Hz): D1=[400,800], D2=[200,400], D3=[100,200], A3=[0,100] Hz
coeffs = pywt.wavedec(noisy_signal, 'db4', level=3)
cA3, cD3, cD2, cD1 = coeffs

fig, axes = plt.subplots(4, 1, figsize=(12, 7))
for ax, c, label in zip(
    axes,
    [cA3, cD3, cD2, cD1],
    ['A3 [0-100 Hz] (Fundamental 50 Hz)', 'D3 [100-200 Hz]', 'D2 [200-400 Hz] (Harmonisa 250 Hz)', 'D1 [400-800 Hz] (Transien 550 Hz di t=0.45s!)'],
):
    ax.plot(c, lw=1.0)
    ax.set_title(label)
plt.tight_layout()
plt.show()

# Soft Thresholding Universal (VisuShrink)
sigma_est = np.median(np.abs(cD1)) / 0.6745
uthresh = sigma_est * np.sqrt(2 * np.log(len(noisy_signal)))
coeffs_clean = [cA3] + [pywt.threshold(c, uthresh, mode='soft') for c in [cD3, cD2, cD1]]
denoised = pywt.waverec(coeffs_clean, 'db4')[:len(noisy_signal)]

snr_before = 10 * np.log10(np.sum((clean_base + transient)**2) / np.sum((noisy_signal - (clean_base + transient))**2))
snr_after = 10 * np.log10(np.sum((clean_base + transient)**2) / np.sum((denoised - (clean_base + transient))**2))
print(f"SNR Sebelum Denoising: {snr_before:.2f} dB | Sesudah Wavelet Denoising: {snr_after:.2f} dB")

## 3. Tugas & Pertanyaan Analisis (C4–C6)
1. **[C4]** Amati koefisien `D1`: pada indeks ke berapa lonjakan transien muncul? Konversikan indeks tersebut kembali ke satuan detik!
2. **[C5]** Bandingkan hasil denoising menggunakan *mother wavelet* `'haar'`, `'db4'`, dan `'sym8'` berdasarkan peningkatan SNR (dB).